# MAIA - FNLP - Proyecto FACULTADES
Clasificación multiclase: dado el resumen de un trabajo académico en español, predecir a cuál de las 5 facultades pertenece.

**Clases:** `administracion_empresas` · `medicina` · `gobierno` · `derecho` · `economia`  
**Métrica:** F1-macro (sin ponderar por clase)  
**Reglas:** solo PLN clásico con scikit-learn · no transformers · Word2Vec/FastText permitidos · test.csv solo para predicciones finales

## 0. Configuración del entorno

In [ ]:
# Clona el repositorio
!git clone https://github.com/luzdvanegasp/kaggle-competition_FNLP.git
%cd kaggle-competition_FNLP

In [ ]:
!pip install -q nltk unidecode

## 1. Carga de librerías

### 1.1 Visualización y análisis de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

### 1.2 Preprocesamiento de texto

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

### 1.3 Vectorización

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

### 1.4 Modelos de clasificación

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

### 1.5 Evaluación y ajuste de hiperparámetros

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import time

### 1.6 Configuración general y carga de datos

In [ ]:
seed = 0
np.random.seed(seed)

train_path = 'inputs/train.csv'
val_path   = 'inputs/val.csv'
test_path  = 'inputs/test.csv'
text_col   = 'resumen'
label_col  = 'facultad'
id_col     = 'id'

train = pd.read_csv(train_path)
val   = pd.read_csv(val_path)
test  = pd.read_csv(test_path)

print(f'Train: {train.shape} | Val: {val.shape} | Test: {test.shape}')
train.head(2)

## 2. Análisis exploratorio (EDA)

In [ ]:
# Distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, df, titulo in zip(axes, [train, val], ['Train (6.000)', 'Val (1.500)']):
    conteo = df[label_col].value_counts()
    conteo.plot(kind='bar', ax=ax, color='navy', edgecolor='white')
    ax.set_title(titulo)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()+5), ha='center', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Longitud de resúmenes
train['n_palabras'] = train[text_col].str.split().str.len()
plt.figure(figsize=(10, 4))
for label in sorted(train[label_col].unique()):
    subset = train[train[label_col]==label]['n_palabras']
    subset.plot(kind='kde', label=f'{label} (med={int(subset.median())})', color='navy' if label == sorted(train[label_col].unique())[0] else None)
plt.legend(fontsize=9)
plt.title('Distribución de longitud por facultad')
plt.xlabel('N° palabras'); plt.tight_layout(); plt.show()
print(train.groupby(label_col)['n_palabras'].describe()[['mean','min','50%','max']].round(0))

## 3. Preprocesamiento de texto

In [ ]:
stopwords_es = set(stopwords.words('spanish'))
extra_stop = {
    'objetivo','objetivos','presente','trabajo','tesis','estudio','investigacion',
    'resultado','resultados','conclusion','conclusiones','metodologia','analisis',
    'mediante','utilizando','utilizar','así','siendo','segun','dicho','dichos','dicha','cual','cuales'
}
stopwords_es.update(extra_stop)

def preprocess(text: str, quitar_sw: bool = True) -> str:
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if quitar_sw:
        tokens = [t for t in text.split() if t not in stopwords_es and len(t) > 2]
        text = ' '.join(tokens)
    return text

# Aplicar solo a train y val (NUNCA a test antes de predecir)
train['texto_proc'] = train[text_col].apply(preprocess)
val['texto_proc']   = val[text_col].apply(preprocess)

print('Original :', train[text_col].iloc[0][:200])
print()
print('Procesado:', train['texto_proc'].iloc[0][:200])

## 4. Baseline — TF-IDF + comparativa de clasificadores

In [ ]:
x_train = train['texto_proc'].values
y_train = train[label_col].values
x_val   = val['texto_proc'].values
y_val   = val[label_col].values

tfidf_params = dict(ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)

modelos = {
    'LogReg C=1' : LogisticRegression(C=1,  max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed),
    'LogReg C=5' : LogisticRegression(C=5,  max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed),
    'LogReg C=10': LogisticRegression(C=10, max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed),
    'LinearSVC'  : LinearSVC(C=1, class_weight='balanced', max_iter=3000, random_state=seed),
    'ComplementNB': ComplementNB(alpha=0.1),
}

resultados = {}
print(f"{'Modelo':<14} {'F1-macro val':>13} {'Tiempo':>8}")
print('-'*38)
for nombre, clf in modelos.items():
    pipe = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', clf)])
    t0 = time.time()
    pipe.fit(x_train, y_train)
    f1 = f1_score(y_val, pipe.predict(x_val), average='macro')
    resultados[nombre] = (f1, pipe)
    print(f"{nombre:<14} {f1:>13.4f} {time.time()-t0:>7.1f}s")

In [ ]:
# Comparación visual
nombres = list(resultados.keys())
f1s     = [resultados[n][0] for n in nombres]

plt.figure(figsize=(8, 4))
bars = plt.barh(nombres, f1s, color='navy', edgecolor='white')
for bar, val_f1 in zip(bars, f1s):
    plt.text(val_f1 + 0.002, bar.get_y() + bar.get_height()/2, f'{val_f1:.4f}', va='center', fontsize=9)
plt.xlabel('F1-macro val')
plt.title('Comparación de modelos baseline')
plt.xlim(0.7, 0.95)
plt.tight_layout(); plt.show()

In [ ]:
mejor_nombre = max(resultados, key=lambda k: resultados[k][0])
mejor_pipe   = resultados[mejor_nombre][1]
print(f'Mejor modelo: {mejor_nombre}  ->  F1-macro val = {resultados[mejor_nombre][0]:.4f}\n')
print(classification_report(y_val, mejor_pipe.predict(x_val)))

## 5. TF-IDF palabras + caracteres

In [ ]:
tfidf_word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=50_000, sublinear_tf=True, min_df=3)

x_tr_combo = hstack([tfidf_word.fit_transform(x_train), tfidf_char.fit_transform(x_train)])
x_va_combo = hstack([tfidf_word.transform(x_val),       tfidf_char.transform(x_val)])

clf_combo = LogisticRegression(C=5, max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_combo.fit(x_tr_combo, y_train)
f1_combo = f1_score(y_val, clf_combo.predict(x_va_combo), average='macro')
print(f'TF-IDF word+char  ->  F1-macro val = {f1_combo:.4f}')

## 6. Ajuste de hiperparámetros (GridSearch sobre train+val)

In [ ]:
x_all = np.concatenate([x_train, x_val])
y_all = np.concatenate([y_train, y_val])

pipe_gs = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.95)),
    ('clf',   LogisticRegression(class_weight='balanced', max_iter=2000, solver='lbfgs', multi_class='multinomial', random_state=seed))
])

param_grid = {
    'tfidf__ngram_range':  [(1,1),(1,2),(1,3)],
    'tfidf__max_features': [60_000, 100_000, 150_000],
    'clf__C':              [1, 3, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
gs = GridSearchCV(pipe_gs, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)
gs.fit(x_all, y_all)

print(f'\nMejores parametros : {gs.best_params_}')
print(f'Mejor F1-macro CV  : {gs.best_score_:.4f}')

## 7. Análisis de errores

In [ ]:
# Reentrenamos el mejor pipeline solo con train para evaluar en val
gs.best_estimator_.fit(x_train, y_train)
y_pred_val = gs.best_estimator_.predict(x_val)

print(f'F1-macro val: {f1_score(y_val, y_pred_val, average="macro"):.4f}\n')
print(classification_report(y_val, y_pred_val))

clases = gs.best_estimator_.classes_
cm = confusion_matrix(y_val, y_pred_val, labels=clases)
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Matriz de confusion - Val')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

val['pred'] = y_pred_val
errores = val[val[label_col] != val['pred']]
print(f'\nErrores: {len(errores)}/{len(val)} ({len(errores)/len(val)*100:.1f}%)')
print(errores.groupby([label_col,'pred']).size().sort_values(ascending=False).head(8))

## 8. Ensemble (Soft Voting)

In [ ]:
best_ngram = gs.best_params_['tfidf__ngram_range']
best_maxf  = gs.best_params_['tfidf__max_features']
best_c     = gs.best_params_['clf__C']

tfidf_ens = TfidfVectorizer(ngram_range=best_ngram, max_features=best_maxf, sublinear_tf=True, min_df=2, max_df=0.95)
x_tr_ens = tfidf_ens.fit_transform(x_train)
x_va_ens = tfidf_ens.transform(x_val)

clf_lr  = LogisticRegression(C=best_c, max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_svc = CalibratedClassifierCV(LinearSVC(C=1, class_weight='balanced', max_iter=3000, random_state=seed))
clf_nb  = ComplementNB(alpha=0.1)

ens = VotingClassifier([('lr',clf_lr),('svc',clf_svc),('nb',clf_nb)], voting='soft')
ens.fit(x_tr_ens, y_train)
f1_ens = f1_score(y_val, ens.predict(x_va_ens), average='macro')
print(f'Ensemble F1-macro val: {f1_ens:.4f}')

## 9. Entrenamiento final y submission

In [ ]:
# Entrena sobre train + val con el mejor pipeline
modelo_final = gs.best_estimator_
modelo_final.fit(x_all, y_all)
print('Modelo entrenado sobre train + val')

# Preprocesar test SOLO aqui
test['texto_proc'] = test[text_col].apply(preprocess)
y_pred_test = modelo_final.predict(test['texto_proc'].values)

submission = pd.DataFrame({id_col: test[id_col], label_col: y_pred_test})
submission.to_csv('submission.csv', index=False)

print(f'\nDistribucion predicciones:')
print(submission[label_col].value_counts())
submission.head()

In [ ]:
from google.colab import files
files.download('submission.csv')
print('submission.csv listo para subir a Kaggle')